# Text-to-Speech (TTS) with SpeechT5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nontgcob/HKU-InnoWing-STT-TTS-Workshop/blob/main/src/tts.ipynb)

This notebook demonstrates Text-to-Speech (TTS) with Microsoft's SpeechT5 model from Hugging Face Transformers.

The workflow loads the TTS model, prepares text input, selects a speaker embedding, generates a mel spectrogram, and converts that spectrogram into playable speech.


## 1. Install the Required Packages

Run this setup cell in Colab before loading the model components.


In [ ]:
%pip install -q --upgrade "datasets==2.21.0" "fsspec==2024.6.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 12.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


## 2. Load the SpeechT5 Model

We start by loading the processor and base text-to-speech model.


In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")

## 3. Prepare the Text Input

Change the text below to synthesize a different utterance.


In [ ]:
text = "Hello, how are you doing today?"
inputs = processor(text=text, return_tensors="pt", padding=True)

## 4. Load a Speaker Embedding

SpeechT5 uses a speaker embedding to control the voice characteristics of the generated speech.


In [ ]:
from datasets import load_dataset
import torch

embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7000]["xvector"]).unsqueeze(0)

In [ ]:
speaker_embeddings.shape

## 5. Generate and Visualize the Mel Spectrogram

The model first produces a mel spectrogram representation before waveform generation.


In [ ]:
spectrogram = model.generate_speech(inputs["input_ids"], speaker_embeddings)

In [ ]:
import matplotlib.pyplot as plt
import librosa.display

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    spectrogram.numpy(),
    sr=16000,  # 采样率
    x_axis="time",
    y_axis="mel",
)
plt.colorbar(label="Magnitude (dB)", format="%+2.0f")
plt.title("Generated Mel Spectrogram")
plt.tight_layout()
plt.show()

## 6. Convert the Spectrogram to Audio

We use the HiFi-GAN vocoder to turn the generated spectrogram into a waveform you can play back in the notebook.


In [ ]:
from transformers import SpeechT5HifiGan

vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

In [ ]:
speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

In [ ]:
from IPython.display import Audio

Audio(speech.numpy(), rate=16000)